# DL Cluster Ensembles

Ноутбук строит ансамбли поверх tuned DL-моделей с cluster meta-признаками.
Входная таблица берётся из `Dl_cluster.ipynb`, затем модели восстанавливаются и сравниваются в нескольких семействах ансамблей.

In [ ]:
# =========================
# Cluster utilities
# =========================
HEAVY_CLUSTERERS = {
    "AffinityPropagation", "MeanShift", "SpectralClustering",
    "Ward", "AgglomerativeClustering", "DBSCAN", "OPTICS", "HDBSCAN",
}
CLUSTER_MAX_HEAVY_FIT_ROWS = 3000
CLUSTER_MIN_VALID_CLUSTERS = 2
CLUSTER_MAX_VALID_CLUSTERS = 20

def cluster_make_estimator(name, params):
    if name == "MiniBatchKMeans":
        return MiniBatchKMeans(
            n_clusters=params["n_clusters"],
            random_state=SEED,
            n_init=20,
            batch_size=1024,
        )
    elif name == "AffinityPropagation":
        return AffinityPropagation(
            damping=params["damping"],
            random_state=SEED,
        )
    elif name == "MeanShift":
        return MeanShift(
            bandwidth=params["bandwidth"],
        )
    elif name == "SpectralClustering":
        return SpectralClustering(
            n_clusters=params["n_clusters"],
            affinity=params["affinity"],
            random_state=SEED,
            assign_labels="kmeans",
        )
    elif name == "Ward":
        return AgglomerativeClustering(
            n_clusters=params["n_clusters"],
            linkage="ward",
        )
    elif name == "AgglomerativeClustering":
        return AgglomerativeClustering(
            n_clusters=params["n_clusters"],
            linkage=params["linkage"],
        )
    elif name == "DBSCAN":
        return DBSCAN(
            eps=params["eps"],
            min_samples=params["min_samples"],
        )
    elif name == "OPTICS":
        return OPTICS(
            min_samples=params["min_samples"],
            xi=params["xi"],
        )
    elif name == "BIRCH":
        return Birch(
            n_clusters=params["n_clusters"],
            threshold=params["threshold"],
        )
    elif name == "GaussianMixture":
        return GaussianMixture(
            n_components=params["n_components"],
            covariance_type=params["covariance_type"],
            random_state=SEED,
        )
    elif name == "HDBSCAN":
        return hdbscan.HDBSCAN(
            min_cluster_size=params["min_cluster_size"],
            min_samples=params["min_samples"],
            prediction_data=True,
        )
    else:
        raise ValueError(name)

def cluster_build_centroids(X, labels):
    valid_mask = labels != -1
    valid_labels = np.unique(labels[valid_mask])

    if len(valid_labels) < CLUSTER_MIN_VALID_CLUSTERS:
        return None, None

    label_map = {old: new for new, old in enumerate(sorted(valid_labels))}
    mapped_labels = np.full_like(labels, fill_value=-1)

    centroids = []
    for old_lab in sorted(valid_labels):
        mask = labels == old_lab
        mapped_labels[mask] = label_map[old_lab]
        centroids.append(X[mask].mean(axis=0))

    centroids = np.vstack(centroids)

    if len(centroids) > CLUSTER_MAX_VALID_CLUSTERS:
        return None, None

    return centroids, mapped_labels

def cluster_assign_by_centroid(X, centroids):
    labels, dmin = pairwise_distances_argmin_min(X, centroids, metric="euclidean")
    all_d = pairwise_distances(X, centroids, metric="euclidean")
    return labels, dmin, all_d

def cluster_soft_probs_from_dist(all_d):
    inv_d = 1.0 / (all_d + 1e-6)
    return inv_d / inv_d.sum(axis=1, keepdims=True)

def cluster_fit_and_assign(clusterer_name, params, Xtr_embed, Xte_embed):
    Xfit = Xtr_embed
    fit_mode = "full_train"

    if clusterer_name in HEAVY_CLUSTERERS and len(Xtr_embed) > CLUSTER_MAX_HEAVY_FIT_ROWS:
        idx = rng.choice(len(Xtr_embed), size=CLUSTER_MAX_HEAVY_FIT_ROWS, replace=False)
        idx = np.sort(idx)
        Xfit = Xtr_embed[idx]
        fit_mode = f"subsample_{CLUSTER_MAX_HEAVY_FIT_ROWS}"

    est = cluster_make_estimator(clusterer_name, params)

    if clusterer_name == "GaussianMixture":
        est.fit(Xfit)
        tr_labels = est.predict(Xtr_embed)
        te_labels = est.predict(Xte_embed)
        tr_proba = est.predict_proba(Xtr_embed)
        te_proba = est.predict_proba(Xte_embed)
        if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
            return None
        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native"

    if clusterer_name == "HDBSCAN":
        est.fit(Xfit)
        fit_labels = est.labels_
        valid = fit_labels != -1
        if len(np.unique(fit_labels[valid])) < CLUSTER_MIN_VALID_CLUSTERS:
            return None
        try:
            _te_labels_raw, _ = hdbscan.approximate_predict(est, Xte_embed)
        except Exception:
            return None

        centroids, _ = cluster_build_centroids(Xfit, fit_labels)
        if centroids is None:
            return None

        tr_labels, _, tr_dist = cluster_assign_by_centroid(Xtr_embed, centroids)
        te_labels, _, te_dist = cluster_assign_by_centroid(Xte_embed, centroids)
        tr_proba = cluster_soft_probs_from_dist(tr_dist)
        te_proba = cluster_soft_probs_from_dist(te_dist)
        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native_test"

    if clusterer_name == "AffinityPropagation":
        if "preference_quantile" in params:
            sims = -pairwise_distances(Xfit, metric="euclidean")
            q = params["preference_quantile"]
            pref = np.quantile(sims[np.triu_indices_from(sims, k=1)], q)
            est.preference = pref
        fit_labels = est.fit_predict(Xfit)
    elif hasattr(est, "fit_predict"):
        fit_labels = est.fit_predict(Xfit)
    else:
        est.fit(Xfit)
        if hasattr(est, "labels_"):
            fit_labels = est.labels_
        elif hasattr(est, "predict"):
            fit_labels = est.predict(Xfit)
        else:
            return None

    if hasattr(est, "predict") and clusterer_name in {"MiniBatchKMeans", "BIRCH"} and len(Xfit) == len(Xtr_embed):
        tr_labels = est.predict(Xtr_embed)
        te_labels = est.predict(Xte_embed)

        if hasattr(est, "cluster_centers_"):
            tr_d = pairwise_distances(Xtr_embed, est.cluster_centers_, metric="euclidean")
            te_d = pairwise_distances(Xte_embed, est.cluster_centers_, metric="euclidean")
            tr_proba = cluster_soft_probs_from_dist(tr_d)
            te_proba = cluster_soft_probs_from_dist(te_d)
        else:
            tr_proba = te_proba = None

        if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
            return None

        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native"

    centroids, _ = cluster_build_centroids(Xfit, fit_labels)
    if centroids is None:
        return None

    tr_labels, _, tr_d = cluster_assign_by_centroid(Xtr_embed, centroids)
    te_labels, _, te_d = cluster_assign_by_centroid(Xte_embed, centroids)
    tr_proba = cluster_soft_probs_from_dist(tr_d)
    te_proba = cluster_soft_probs_from_dist(te_d)

    if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
        return None

    return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_centroid"

def cluster_build_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    tr_feat = pd.DataFrame({"cluster_id": tr_labels})
    te_feat = pd.DataFrame({"cluster_id": te_labels})

    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat["cluster_size_train"] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat["cluster_size_train"] = pd.Series(te_labels).map(tr_sizes).fillna(0)

    tr_feat["is_noise"] = (tr_feat["cluster_id"] == -1).astype(int)
    te_feat["is_noise"] = (te_feat["cluster_id"] == -1).astype(int)

    tr_ohe = pd.get_dummies(tr_feat["cluster_id"], prefix="cluster")
    te_ohe = pd.get_dummies(te_feat["cluster_id"], prefix="cluster")
    te_ohe = te_ohe.reindex(columns=tr_ohe.columns, fill_value=0)

    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)

    if tr_proba is not None and te_proba is not None:
        for k in range(tr_proba.shape[1]):
            tr_feat[f"cluster_proba_{k}"] = tr_proba[:, k]
            te_feat[f"cluster_proba_{k}"] = te_proba[:, k]

    return tr_feat, te_feat

print("Cluster utilities defined.")

In [ ]:
from pathlib import Path
import os
import re
import json
import ast
import math
import gc
import random
import time
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from scipy.optimize import nnls, minimize
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, median_absolute_error,
    pairwise_distances, pairwise_distances_argmin_min,
)
from sklearn.cluster import (
    MiniBatchKMeans, AffinityPropagation, MeanShift,
    SpectralClustering, AgglomerativeClustering, DBSCAN, OPTICS, Birch,
)
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import (
    LinearRegression, RidgeCV, LassoCV, ElasticNetCV,
    BayesianRidge, HuberRegressor, QuantileRegressor,
)
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGB_FOR_STACKING = True
except Exception:
    HAS_XGB_FOR_STACKING = False

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except Exception:
    HDBSCAN_AVAILABLE = False

warnings_filter = __import__("warnings")
warnings_filter.filterwarnings("ignore")

SEED = 2
DURATION_CAP = 960
TARGET_COL = "duration_hours"

def seed_everything(seed: int = 2):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(SEED)
rng = np.random.RandomState(SEED)

RUN_NAME = datetime.now().strftime("run_%Y%m%d_%H%M%S")
BLEND_FIT_FRAC = 0.50
FORCE_REFRESH_BASE_PREDICTIONS = False
RESUME_IF_PREDICTIONS_EXIST = True

RUN_ALL_PAIRS = True
RUN_ALL_TRIPLES = True
RUN_TOPK_PREFIX_ENSEMBLES = True
RUN_GREEDY_SEARCH = True
RUN_STACKERS = True
RUN_EXHAUSTIVE_TOPN_SUBSETS = True
EXHAUSTIVE_TOP_N = 10
REFIT_TOP_ENSEMBLES = 150
PAIR_WEIGHT_GRID = np.linspace(0.0, 1.0, 101)

FULLFIT_EPOCHS_MIN = 40
FULLFIT_EPOCHS_CAP = 150
FULLFIT_EPOCHS_DEFAULT = 100
FULLFIT_GROWNET_EPOCHS_PER_STAGE = 50

DATA_PATH_ENV = "DATA_FINALL_PATH"
CLUSTER_TUNED_SUMMARY_PATH_ENV = "DL_CLUSTER_TUNED_SUMMARY_PATH"
ARTIFACTS_DIR_ENV = "DL_CLUSTER_ENSEMBLES_ARTIFACTS_DIR"


def require_path(env_name: str, label: str) -> Path:
    value = os.environ.get(env_name)
    if not value:
        raise RuntimeError(f"Укажи путь к {label} в переменной окружения {env_name}.")
    path = Path(value).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"Файл для {label} не найден: {path}")
    return path


DATA_PATH = require_path(DATA_PATH_ENV, "data_finall")
CLUSTER_TUNED_SUMMARY_PATH = require_path(CLUSTER_TUNED_SUMMARY_PATH_ENV, "cluster tuned summary")

ART_DIR = Path(os.environ.get(ARTIFACTS_DIR_ENV, "./artifacts_dl_cluster_ensembles_top10")).expanduser()
ART_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = ART_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

PRED_CACHE_DIR = ART_DIR / "pred_cache"
for _sub in ["split_val", "split_test_typical", "split_test_full", "fullfit_test_typical", "fullfit_test_full"]:
    (PRED_CACHE_DIR / _sub).mkdir(parents=True, exist_ok=True)

print(f"Cluster tuned rows source ready: {CLUSTER_TUNED_SUMMARY_PATH.name}")

# =========================
# Helpers
# =========================
def parse_maybe_dict(raw):
    if isinstance(raw, dict):
        return raw
    if raw is None:
        return {}
    try:
        if pd.isna(raw):
            return {}
    except Exception:
        pass
    txt = str(raw).strip()
    if txt == "" or txt.lower() == "nan":
        return {}
    # JSON first
    try:
        return json.loads(txt)
    except Exception:
        pass
    # Safe fallback for csv-dumped python dicts with nan/NaN/inf
    safe = re.sub(r'(?<!["\'])\bNaN\b(?!["\'])', 'None', txt)
    safe = re.sub(r'(?<!["\'])\bnan\b(?!["\'])', 'None', safe)
    safe = re.sub(r'(?<!["\'])\binf\b(?!["\'])', 'None', safe)
    safe = re.sub(r'(?<!["\'])\b-Infinity\b(?!["\'])', 'None', safe)
    safe = re.sub(r'(?<!["\'])\bInfinity\b(?!["\'])', 'None', safe)
    return ast.literal_eval(safe)

def sanitize_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", name)

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, None)

def calc_reg_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(float)
    y_pred = clip_pred(y_pred)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MdAE": float(median_absolute_error(y_true, y_pred)),
    }

# =========================
# Load tuned summary
# =========================
tuned_df = pd.read_csv(CLUSTER_TUNED_SUMMARY_PATH)
required_cols = {"model", "clusterer", "cluster_params", "best_params"}
missing_cols = required_cols - set(tuned_df.columns)
if missing_cols:
    raise ValueError(f"В tuned summary не хватает колонок: {sorted(missing_cols)}")

sort_candidates = [c for c in ["tuned_cv_MAE", "screening_val_MAE", "tuned_holdout_full_MAE"] if c in tuned_df.columns]
if sort_candidates:
    tuned_df = tuned_df.sort_values(sort_candidates[0], kind="stable").reset_index(drop=True)

tuned_df = tuned_df.drop_duplicates(subset=["model"], keep="first").head(10).reset_index(drop=True)
SELECTED_MODELS = tuned_df["model"].tolist()

if len(SELECTED_MODELS) < 2:
    raise RuntimeError("Для ансамблей нужно минимум 2 модели в tuned summary.")

display(tuned_df)
tuned_df.to_csv(RUN_DIR / "cluster_top10_tuned_summary_used.csv", index=False)
tuned_df.to_csv(ART_DIR / "cluster_top10_tuned_summary_used_latest.csv", index=False)

# =========================
# Load dataset and reproduce cluster notebook splits
# =========================
df = pd.read_csv(DATA_PATH)

if TARGET_COL not in df.columns:
    raise ValueError(f"В датасете нет целевой колонки {TARGET_COL!r}")

# Важно: повторяем логику исходного cluster-ноутбука.
split_idx = int(len(df) * 0.8)
train_raw = df.iloc[:split_idx].copy().reset_index(drop=True)
holdout_raw = df.iloc[split_idx:].copy().reset_index(drop=True)

train_typical = train_raw[train_raw[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)
holdout_typical = holdout_raw[holdout_raw[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)

# Только numeric-признаки, как в 08-ноутбуке
Xtrain = train_typical.drop(columns=[TARGET_COL], errors="ignore").select_dtypes(include=["number"]).reset_index(drop=True)
ytrain = train_typical[TARGET_COL].reset_index(drop=True)

Xholdout_full = holdout_raw.drop(columns=[TARGET_COL], errors="ignore").select_dtypes(include=["number"]).reset_index(drop=True)
yholdout_full = holdout_raw[TARGET_COL].reset_index(drop=True)

Xholdout_typical = holdout_typical.drop(columns=[TARGET_COL], errors="ignore").select_dtypes(include=["number"]).reset_index(drop=True)
yholdout_typical = holdout_typical[TARGET_COL].reset_index(drop=True)

print(f"Rows total: {len(df)} | train_raw: {len(train_raw)} | train_typical: {len(train_typical)}")
print(f"holdout_typical: {len(holdout_typical)} | holdout_full: {len(holdout_raw)}")
print(f"Xtrain shape (numeric only): {Xtrain.shape}")

# inner split inside train_typical
val_cut = int(len(Xtrain) * 0.8)

screen_Xtr0 = Xtrain.iloc[:val_cut].copy().reset_index(drop=True)
screen_ytr0 = ytrain.iloc[:val_cut].values.astype(np.float32)

screen_Xva0 = Xtrain.iloc[val_cut:].copy().reset_index(drop=True)
screen_yva0 = ytrain.iloc[val_cut:].values.astype(np.float32)

cluster_Xtr0 = Xtrain.copy().reset_index(drop=True)
cluster_ytr0 = ytrain.values.astype(np.float32)

cluster_Xte0 = Xholdout_full.copy().reset_index(drop=True)
cluster_yte0 = yholdout_full.values.astype(np.float32)

cluster_Xtyp0 = Xholdout_typical.copy().reset_index(drop=True)
cluster_ytyp0 = yholdout_typical.values.astype(np.float32)

# clustering preprocessing
cluster_scaler = StandardScaler()
cluster_Xtr_scaled = cluster_scaler.fit_transform(cluster_Xtr0)
cluster_Xte_scaled = cluster_scaler.transform(cluster_Xte0)
cluster_Xtyp_scaled = cluster_scaler.transform(cluster_Xtyp0)

screen_Xtr_scaled = cluster_scaler.transform(screen_Xtr0)
screen_Xva_scaled = cluster_scaler.transform(screen_Xva0)

cluster_pca_n_components = min(30, cluster_Xtr0.shape[1])
cluster_pca = PCA(n_components=cluster_pca_n_components, random_state=SEED)
cluster_Xtr_embed = cluster_pca.fit_transform(cluster_Xtr_scaled)
cluster_Xte_embed = cluster_pca.transform(cluster_Xte_scaled)
cluster_Xtyp_embed = cluster_pca.transform(cluster_Xtyp_scaled)

screen_Xtr_embed = cluster_pca.transform(screen_Xtr_scaled)
screen_Xva_embed = cluster_pca.transform(screen_Xva_scaled)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else
    "cpu"
)
INPUT_DIM = Xtrain.shape[1]
print(f"Device: {device} | INPUT_DIM: {INPUT_DIM} | HDBSCAN available: {HDBSCAN_AVAILABLE}")

# aliases for ensemble stage
y_dl_va = screen_yva0.copy()
ytest_typical = cluster_ytyp0.copy()
ytest_full = cluster_yte0.copy()

In [ ]:
# ── 2.1  MLP ─────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, d_in, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


# ── 2.2  ResMLP ──────────────────────────────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.block(x))

class ResMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU())
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))


# ── 2.3  SNN ─────────────────────────────────────────────────

class SNN(nn.Module):
    def __init__(self, d_in, hidden_dims=[256, 128], alpha_dropout=0.1):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.SELU(), nn.AlphaDropout(alpha_dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)


# ── 2.4  GatedMLP ───────────────────────────────────────────

class GatedBlock(nn.Module):
    def __init__(self, d_in, d_out, dropout):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out * 2)
        self.bn = nn.BatchNorm1d(d_out)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.linear(x)
        h1, h2 = h.chunk(2, dim=-1)
        return self.drop(self.bn(h1 * torch.sigmoid(h2)))

class GatedMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        blocks = [GatedBlock(d_in, hidden, dropout)]
        for _ in range(n_blocks - 1):
            blocks.append(GatedBlock(hidden, hidden, dropout))
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(x))


# ── 2.5  GANDALF ─────────────────────────────────────────────

class GANDALF(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.gate_fc = nn.Linear(d_in, d_in)
        layers = [nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        for _ in range(n_blocks - 1):
            layers += [nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        self.dnn = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        gate = torch.sigmoid(self.gate_fc(x))
        return self.head(self.dnn(x * gate))


# ── 2.6  DAE-MLP ────────────────────────────────────────────

class DAEMLP(nn.Module):
    def __init__(self, d_in, hidden=256, latent=64, noise=0.3, dropout=0.3):
        super().__init__()
        self.noise = noise
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, latent), nn.BatchNorm1d(latent), nn.SiLU(),
        )
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.SiLU(), nn.Linear(hidden, d_in))
        self.reg_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(latent, 32), nn.SiLU(), nn.Linear(32, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x_in = x * (torch.rand_like(x) > self.noise).float() if self.training and self.noise > 0 else x
        z = self.encoder(x_in)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(z), x)
        return self.reg_head(z)


# ── 2.7  CNN1D ───────────────────────────────────────────────

class CNN1D(nn.Module):
    def __init__(self, d_in, channels=[64, 128, 64], ks=5, dropout=0.3):
        super().__init__()
        layers = []
        in_ch = 1
        for ch in channels:
            layers += [nn.Conv1d(in_ch, ch, ks, padding=ks // 2),
                       nn.BatchNorm1d(ch), nn.SiLU(), nn.Dropout(dropout)]
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Linear(channels[-1], 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.conv(x))
        return self.head(x.squeeze(-1))


# ── 2.8  BiGRU ───────────────────────────────────────────────

class BiGRU(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden, num_layers=n_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        _, h = self.gru(x.unsqueeze(-1))
        return self.head(torch.cat([h[-2], h[-1]], dim=1))


# ── 2.9  FT-Transformer ─────────────────────────────────────

class FTTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.10  TabTransformer ────────────────────────────────────

class TabTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.col_emb = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_in * d_model + d_in, mlp_hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1)) + self.col_emb
        out = self.transformer(tok)
        pooled = out.reshape(out.size(0), -1)
        return self.head(torch.cat([pooled, x], dim=1))


# ── 2.11  SAINT ──────────────────────────────────────────────

class SAINTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.sa_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ia_norm = nn.LayerNorm(d_model)
        self.inter_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model * 4),
                                 nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
    def forward(self, x):
        h = self.sa_norm(x)
        h, _ = self.self_attn(h, h, h)
        x = x + h
        B, F_, D = x.shape
        if B > 1 and self.training:
            xt = x.permute(1, 0, 2)
            h2 = self.ia_norm(xt)
            h2, _ = self.inter_attn(h2, h2, h2)
            x = (xt + h2).permute(1, 0, 2)
        return x + self.ffn(x)

class SAINT(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        self.blocks = nn.ModuleList([SAINTBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        for blk in self.blocks:
            tok = blk(tok)
        return self.head(tok[:, 0])


# ── 2.12  AutoInt ────────────────────────────────────────────

class AutoIntLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.W_res = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        h, _ = self.attn(x, x, x)
        return F.relu(self.norm(h + self.W_res(x)))

class AutoInt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([AutoIntLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in * d_model, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        for layer in self.layers:
            tok = layer(tok)
        return self.head(tok.reshape(tok.size(0), -1))


# ── 2.13  Trompt ─────────────────────────────────────────────

class TromptLayer(nn.Module):
    def __init__(self, d_in, d_model, n_heads, dropout):
        super().__init__()
        self.prompt = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.gate = nn.Linear(d_model, 1)
    def forward(self, feat_emb, x_raw):
        B = feat_emb.shape[0]
        prompted = feat_emb + self.prompt.expand(B, -1, -1)
        h, _ = self.attn(prompted, prompted, prompted)
        h = self.norm1(prompted + h)
        h = self.norm2(h + self.ffn(h))
        weights = torch.softmax(self.gate(h).squeeze(-1), dim=1)
        return h, weights * x_raw

class Trompt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([TromptLayer(d_in, d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        preds = []
        for layer in self.layers:
            tok, lp = layer(tok, x)
            preds.append(lp)
        return self.head(torch.stack(preds).mean(0))


# ── 2.14  ExcelFormer ───────────────────────────────────────

class ExcelFormer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.importance = nn.Parameter(torch.zeros(d_in))
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        mask = torch.sigmoid(self.importance)
        x_masked = x * mask
        tok = self.feat_emb(x_masked.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.15  ARM-Net ────────────────────────────────────────────

class ARMNet(nn.Module):
    def __init__(self, d_in, n_exp=64, hidden=128, order=2, dropout=0.2):
        super().__init__()
        self.order = order
        self.exp_layers = nn.ModuleList([nn.Linear(d_in, n_exp) for _ in range(order)])
        self.gate = nn.Sequential(nn.Linear(d_in, n_exp * order), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.BatchNorm1d(n_exp * order), nn.Linear(n_exp * order, hidden),
            nn.SiLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x):
        interactions = []
        for i, layer in enumerate(self.exp_layers):
            h = layer(x)
            if i > 0:
                h = h * interactions[-1]
            interactions.append(F.softplus(h))
        cat = torch.cat(interactions, dim=1)
        return self.head(cat * self.gate(x))


# ── 2.16  NODE ───────────────────────────────────────────────

class NODE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_weights = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.01)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        choices = torch.sigmoid(torch.einsum('bj,tdj->btd', x, self.feat_weights) - self.thresholds)
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        return self.drop(tree_out).mean(dim=-1, keepdim=True)


# ── 2.17  GRANDE ─────────────────────────────────────────────

class GRANDE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_logits = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.tree_weights = nn.Parameter(torch.ones(n_trees) / n_trees)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        feat_sel = F.softmax(self.feat_logits, dim=-1)
        proj = torch.einsum('bj,tdj->btd', x, feat_sel)
        choices = torch.sigmoid(10.0 * (proj - self.thresholds))
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        w = F.softmax(self.tree_weights, dim=0)
        return self.drop((tree_out * w).sum(dim=-1, keepdim=True))


# ── 2.18  Net-DNF ────────────────────────────────────────────

class NetDNF(nn.Module):
    def __init__(self, d_in, n_formulas=64, n_conjuncts=6, dropout=0.2):
        super().__init__()
        self.feat_sel = nn.Parameter(torch.randn(n_formulas, n_conjuncts, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_formulas, n_conjuncts))
        self.formula_w = nn.Linear(n_formulas, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        sel = torch.sigmoid(self.feat_sel)
        proj = torch.einsum('bj,fcj->bfc', x, sel)
        conjuncts = torch.sigmoid(proj - self.thresholds)
        formulas = conjuncts.prod(dim=-1)
        return self.formula_w(self.drop(formulas))


# ── 2.19  TabNet (simplified) ───────────────────────────────

class TabNet(nn.Module):
    def __init__(self, d_in, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1):
        super().__init__()
        self.n_steps = n_steps
        self.relax = relax
        self.bn_init = nn.BatchNorm1d(d_in)
        self.shared_fc = nn.Linear(d_in, n_d + n_a)
        self.step_fcs = nn.ModuleList([nn.Linear(d_in, n_d + n_a) for _ in range(n_steps)])
        self.attn_fcs = nn.ModuleList([nn.Sequential(nn.Linear(n_a, d_in), nn.BatchNorm1d(d_in)) for _ in range(n_steps)])
        self.head = nn.Linear(n_d, 1)
        self.n_d = n_d
        self.drop = nn.Dropout(dropout)
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x = self.bn_init(x)
        prior_scales = torch.ones(x.shape[0], x.shape[1], device=x.device)
        agg = torch.zeros(x.shape[0], self.n_d, device=x.device)
        entropy_loss = 0.0
        for i in range(self.n_steps):
            h = self.shared_fc(x * prior_scales) + self.step_fcs[i](x * prior_scales)
            h_d, h_a = h[:, :self.n_d], h[:, self.n_d:]
            h_d = F.silu(h_d)
            agg = agg + self.drop(h_d)
            attn = self.attn_fcs[i](h_a)
            attn = F.softmax(attn * prior_scales, dim=-1)
            prior_scales = prior_scales * (self.relax - attn)
            entropy_loss += (-attn * torch.log(attn + 1e-15)).mean()
        if self.training:
            self.aux_loss = entropy_loss / self.n_steps
        return self.head(agg)


# ── 2.20  TabR ───────────────────────────────────────────────

class TabR(nn.Module):
    def __init__(self, d_in, hidden=256, n_layers=3, k=32, dropout=0.3):
        super().__init__()
        self.k = k
        layers = []
        prev = d_in
        for i in range(n_layers):
            h = hidden if i == 0 else hidden // 2
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*layers)
        self.latent_dim = prev
        self.retrieval_head = nn.Sequential(nn.Linear(prev * 2, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))
        self.direct_head = nn.Linear(prev, 1)
        self._store_z = None
        self._store_y = None
    def build_store(self, x_train, y_train):
        self.eval()
        with torch.no_grad():
            self._store_z = self.encoder(x_train)
            self._store_y = y_train
    def forward(self, x):
        z = self.encoder(x)
        if self._store_z is not None and not self.training:
            dists = torch.cdist(z, self._store_z)
            _, idx = dists.topk(self.k, largest=False)
            nbr_z = self._store_z[idx]
            sim = -dists.gather(1, idx)
            attn = torch.softmax(sim, dim=1).unsqueeze(-1)
            context = (attn * nbr_z).sum(dim=1)
            return self.retrieval_head(torch.cat([z, context], dim=1))
        return self.direct_head(z)


# ── 2.21  GrowNet stages ────────────────────────────────────

class GrowNetStage(nn.Module):
    def __init__(self, d_in, hidden=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in + 1, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
    def forward(self, x, prev_pred):
        return self.net(torch.cat([x, prev_pred], dim=1))


# ── 2.22  SwitchTab ──────────────────────────────────────────

class SwitchTab(nn.Module):
    def __init__(self, d_in, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, d_enc), nn.BatchNorm1d(d_enc), nn.SiLU(), nn.Dropout(dropout))
        self.mutual_proj = nn.Linear(d_enc, d_mutual)
        self.salient_proj = nn.Linear(d_enc, d_salient)
        self.decoder = nn.Sequential(nn.Linear(d_mutual + d_salient, d_in))
        self.head = nn.Sequential(nn.Linear(d_mutual + d_salient, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        h = self.encoder(x)
        mutual = self.mutual_proj(h)
        salient = self.salient_proj(h)
        rep = torch.cat([mutual, salient], dim=1)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(rep), x)
        return self.head(rep)


# ── 2.23  PTaRL ──────────────────────────────────────────────

class PTaRL(nn.Module):
    def __init__(self, d_in, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_latent), nn.BatchNorm1d(d_latent), nn.SiLU())
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, d_latent) * 0.1)
        self.head = nn.Sequential(nn.Linear(d_latent * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        z = self.encoder(x)
        sim = F.cosine_similarity(z.unsqueeze(1), self.prototypes.unsqueeze(0), dim=-1)
        attn = F.softmax(sim * 10, dim=1)
        proto_rep = (attn.unsqueeze(-1) * self.prototypes.unsqueeze(0)).sum(dim=1)
        return self.head(torch.cat([z, proto_rep], dim=1))


# ── 2.24  DCN v2 ────────────────────────────────────────────

class CrossLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.b = nn.Parameter(torch.zeros(d))
    def forward(self, x0, x):
        return x0 * self.W(x) + self.b + x

class DCNv2(nn.Module):
    def __init__(self, d_in, n_cross=3, hidden=256, dropout=0.3):
        super().__init__()
        self.cross_layers = nn.ModuleList([CrossLayer(d_in) for _ in range(n_cross)])
        self.deep = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.SiLU(), nn.Dropout(dropout))
        self.head = nn.Linear(d_in + hidden // 2, 1)
    def forward(self, x):
        x_cross = x
        for cl in self.cross_layers:
            x_cross = cl(x, x_cross)
        return self.head(torch.cat([x_cross, self.deep(x)], dim=1))


# ═══════════════════════════════════════════════════════════════
#  Baseline MAE для каждой DL-модели (из предыдущего прогона)
#  Нужен для расчёта delta = baseline - cluster_mae
# ═══════════════════════════════════════════════════════════════

dl_baseline_mae = {
    "MLP":            246.89,
    "ResMLP":         245.32,
    "SNN":            252.41,
    "GatedMLP":       247.15,
    "GANDALF":        246.53,
    "DAE-MLP":        253.78,
    "CNN1D":          251.64,
    "BiGRU":          254.92,
    "FT-Transformer": 248.37,
    "TabTransformer": 249.81,
    "SAINT":          250.14,
    "AutoInt":        248.96,
    "Trompt":         251.23,
    "ExcelFormer":    249.45,
    "ARM-Net":        250.67,
    "NODE":           247.82,
    "GRANDE":         248.11,
    "Net-DNF":        252.03,
    "TabNet":         249.58,
    "TabR":           246.41,
    "GrowNet":        247.29,
    "SwitchTab":      251.87,
    "PTaRL":          250.93,
    "DCNv2":          247.64,
}


# ═══════════════════════════════════════════════════════════════
#  Функции обучения на аугментированных данных
# ═══════════════════════════════════════════════════════════════

def train_model_aug(model, X_tr, y_tr, X_va, y_va,
                    epochs=300, lr=1e-3, wd=1e-4, bs=256,
                    patience=30, aux_w=0.0):
    """Обучение с early stopping на произвольных данных (не глобальных)."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_tr, y_tr)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    X_v, y_v = X_va.to(device), y_va.to(device)

    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v_mae = loss_fn(model(X_v), y_v).item()
        sched.step(v_mae)
        if v_mae < best_val:
            best_val = v_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, ep


def train_grownet_aug(X_tr, y_tr, X_va, y_va, X_te,
                      n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                      step_size=0.1, bs=256, patience=30, dropout=0.1):
    """GrowNet на произвольных данных."""
    d_in = X_tr.shape[1]
    stages = []
    X_v, y_v = X_va.to(device), y_va.to(device)
    ds = TensorDataset(X_tr, y_tr)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    best_val, best_n = float("inf"), 0
    for stage_idx in range(n_stages):
        model = GrowNetStage(d_in, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
        best_stage_val, best_stage_state, wait = float("inf"), None, 0
        for ep in range(1, 201):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                prev_v = ensemble_pred(X_v, stages)
                v_pred = prev_v + step_size * model(X_v, prev_v)
                v_mae = loss_fn(v_pred, y_v).item()
            sched.step(v_mae)
            if v_mae < best_stage_val:
                best_stage_val = v_mae
                best_stage_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break
        model.load_state_dict(best_stage_state)
        stages.append(model)
        with torch.no_grad():
            ens_val = loss_fn(ensemble_pred(X_v, stages), y_v).item()
        if ens_val < best_val:
            best_val = ens_val
            best_n = len(stages)
    stages = stages[:best_n]
    return stages, best_val


def make_dl_builders(d_in):
    """Фабрики всех 24 архитектур для заданной размерности."""
    return {
        "MLP":            lambda: MLP(d_in, [256, 128], dropout=0.3),
        "ResMLP":         lambda: ResMLP(d_in, hidden=256, n_blocks=3, dropout=0.3),
        "SNN":            lambda: SNN(d_in, hidden_dims=[256, 128], alpha_dropout=0.1),
        "GatedMLP":       lambda: GatedMLP(d_in, hidden=256, n_blocks=3, dropout=0.3),
        "GANDALF":        lambda: GANDALF(d_in, hidden=256, n_blocks=3, dropout=0.3),
        "DAE-MLP":        lambda: DAEMLP(d_in, hidden=256, latent=64, noise=0.3, dropout=0.3),
        "CNN1D":          lambda: CNN1D(d_in, channels=[64, 128, 64], ks=5, dropout=0.3),
        "BiGRU":          lambda: BiGRU(d_in, hidden=64, n_layers=2, dropout=0.3),
        "FT-Transformer": lambda: FTTransformer(d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
        "TabTransformer": lambda: TabTransformer(d_in, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2),
        "SAINT":          lambda: SAINT(d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
        "AutoInt":        lambda: AutoInt(d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
        "Trompt":         lambda: Trompt(d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
        "ExcelFormer":    lambda: ExcelFormer(d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
        "ARM-Net":        lambda: ARMNet(d_in, n_exp=64, hidden=128, order=2, dropout=0.2),
        "NODE":           lambda: NODE(d_in, n_trees=32, depth=4, dropout=0.0),
        "GRANDE":         lambda: GRANDE(d_in, n_trees=32, depth=4, dropout=0.0),
        "Net-DNF":        lambda: NetDNF(d_in, n_formulas=64, n_conjuncts=6, dropout=0.2),
        "TabNet":         lambda: TabNet(d_in, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1),
        "TabR":           lambda: TabR(d_in, hidden=256, n_layers=3, k=32, dropout=0.3),
        "SwitchTab":      lambda: SwitchTab(d_in, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3),
        "PTaRL":          lambda: PTaRL(d_in, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3),
        "DCNv2":          lambda: DCNv2(d_in, n_cross=3, hidden=256, dropout=0.3),
    }

DL_AUX_W = {"DAE-MLP": 0.1, "TabNet": 0.01, "SwitchTab": 0.1}

print("DL-архитектуры и функции обучения определены.")

In [ ]:
def build_dl_cluster_model(dl_name, d_in, params):
    dropout = params.get("dropout", 0.3)

    if dl_name == "MLP":
        n_layers = params["n_layers"]
        first = params["first_hidden"]
        dims = [max(first // (2 ** i), 32) for i in range(n_layers)]
        return MLP(d_in, dims, dropout=dropout)

    elif dl_name == "ResMLP":
        return ResMLP(
            d_in,
            hidden=params["hidden"],
            n_blocks=params["n_blocks"],
            dropout=dropout,
        )

    elif dl_name == "SNN":
        n_layers = params["n_layers"]
        first = params["first_hidden"]
        dims = [max(first // (2 ** i), 32) for i in range(n_layers)]
        return SNN(
            d_in,
            hidden_dims=dims,
            alpha_dropout=params["alpha_dropout"],
        )

    elif dl_name == "GatedMLP":
        return GatedMLP(
            d_in,
            hidden=params["hidden"],
            n_blocks=params["n_blocks"],
            dropout=dropout,
        )

    elif dl_name == "GANDALF":
        return GANDALF(
            d_in,
            hidden=params["hidden"],
            n_blocks=params["n_blocks"],
            dropout=dropout,
        )

    elif dl_name == "DAE-MLP":
        return DAEMLP(
            d_in,
            hidden=params["hidden"],
            latent=params["latent"],
            noise=params["noise"],
            dropout=dropout,
        )

    elif dl_name == "CNN1D":
        n_conv = params["n_conv"]
        base_ch = params["base_ch"]
        chs = [base_ch * (2 ** i) for i in range(n_conv // 2 + 1)]
        chs += [base_ch * (2 ** i) for i in range(n_conv // 2 - 1, -1, -1)]
        chs = chs[:n_conv]
        return CNN1D(
            d_in,
            channels=chs,
            ks=params["ks"],
            dropout=dropout,
        )

    elif dl_name == "BiGRU":
        return BiGRU(
            d_in,
            hidden=params["hidden"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "FT-Transformer":
        return FTTransformer(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "TabTransformer":
        return TabTransformer(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            mlp_hidden=params["mlp_hidden"],
            dropout=dropout,
        )

    elif dl_name == "SAINT":
        return SAINT(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "AutoInt":
        return AutoInt(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "Trompt":
        return Trompt(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "ExcelFormer":
        return ExcelFormer(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "ARM-Net":
        return ARMNet(
            d_in,
            n_exp=params["n_exp"],
            hidden=params["hidden"],
            order=params["order"],
            dropout=dropout,
        )

    elif dl_name == "NODE":
        return NODE(
            d_in,
            n_trees=params["num_trees"],
            depth=params["depth"],
            dropout=dropout,
        )

    elif dl_name == "GRANDE":
        return GRANDE(
            d_in,
            n_trees=params["num_trees"],
            depth=params["depth"],
            dropout=dropout,
        )

    elif dl_name == "Net-DNF":
        return NetDNF(
            d_in,
            n_formulas=params["n_formulas"],
            n_conjuncts=params["n_conjuncts"],
            dropout=dropout,
        )

    elif dl_name == "TabNet":
        return TabNet(
            d_in,
            n_steps=params["n_steps"],
            n_d=params["n_d"],
            n_a=params["n_a"],
            dropout=dropout,
        )

    elif dl_name == "TabR":
        return TabR(
            d_in,
            hidden=params["hidden"],
            n_layers=params["n_layers"],
            k=params["k"],
            dropout=dropout,
        )

    elif dl_name == "SwitchTab":
        return SwitchTab(
            d_in,
            d_enc=params["d_enc"],
            d_mutual=params["d_mutual"],
            d_salient=params["d_salient"],
            dropout=dropout,
        )

    elif dl_name == "PTaRL":
        return PTaRL(
            d_in,
            n_prototypes=params["n_prototypes"],
            d_latent=params["d_latent"],
            hidden=params["hidden"],
            dropout=dropout,
        )

    elif dl_name == "DCNv2":
        return DCNv2(
            d_in,
            n_cross=params["n_cross"],
            hidden=params["hidden"],
            dropout=dropout,
        )

    else:
        raise ValueError(dl_name)




# =========================
# Full-fit helpers for clusterized DL models
# =========================
def train_model_fullfit_aug(model, X_full, y_full, epochs=100, lr=1e-3, wd=1e-4, bs=256, aux_w=0.0):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(int(epochs), 1), eta_min=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_full, y_full)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)

    for _ in range(max(int(epochs), 1)):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()

    return model

def train_grownet_fullfit_aug(X_full, y_full, n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                              step_size=0.1, bs=256, epochs_per_stage=50, dropout=0.1):
    d_in = X_full.shape[1]
    stages = []
    ds = TensorDataset(X_full, y_full)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    for _stage_idx in range(int(n_stages)):
        model = GrowNetStage(d_in, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(int(epochs_per_stage), 1), eta_min=1e-6)

        for _ in range(max(int(epochs_per_stage), 1)):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()

        stages.append(model)

    return stages

def torch_pred(model, X_tensor):
    model.eval()
    with torch.no_grad():
        pred = model(X_tensor.to(device)).cpu().numpy().flatten()
    return clip_pred(pred)

def grownet_pred(stages, X_tensor, step_size=0.1):
    X_t = X_tensor.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    return clip_pred(p.cpu().numpy().flatten())

def _pred_path(kind: str, model_name: str) -> Path:
    return PRED_CACHE_DIR / kind / f"{sanitize_name(model_name)}.npy"

def _save_pred(kind: str, model_name: str, arr):
    np.save(_pred_path(kind, model_name), np.asarray(arr, dtype=np.float32))

def _load_pred(kind: str, model_name: str):
    path = _pred_path(kind, model_name)
    if not path.exists():
        return None
    return np.load(path)

def _have_all_cached(model_name: str) -> bool:
    needed = [
        _pred_path("split_val", model_name),
        _pred_path("split_test_typical", model_name),
        _pred_path("split_test_full", model_name),
        _pred_path("fullfit_test_typical", model_name),
        _pred_path("fullfit_test_full", model_name),
    ]
    return all(p.exists() for p in needed)

def load_prediction_frame(kind: str, model_names):
    cols = {}
    for m in model_names:
        arr = _load_pred(kind, m)
        if arr is not None:
            cols[m] = arr
    if not cols:
        return pd.DataFrame()
    return pd.DataFrame(cols)

def get_base_leaderboard_from_predictions(val_pred_df: pd.DataFrame, y_val):
    rows = []
    for m in val_pred_df.columns:
        rows.append({
            "model": m,
            "val_MAE_computed": float(mean_absolute_error(y_val, val_pred_df[m].values)),
            "val_RMSE_computed": float(np.sqrt(mean_squared_error(y_val, val_pred_df[m].values))),
        })
    return pd.DataFrame(rows).sort_values("val_MAE_computed", kind="stable").reset_index(drop=True)

def fit_and_predict_tuned_cluster_model(row: pd.Series, data: dict):
    arch = row["model"]
    best_params = parse_maybe_dict(row["best_params"])

    lr = float(best_params.get("lr", 1e-3))
    wd = float(best_params.get("wd", 1e-4))
    bs = int(best_params.get("bs", 256))

    seed_everything(SEED)
    t0 = time.time()

    print(f"\n=== {arch} ===")
    print("clusterer:", row["clusterer"])
    

    if arch == "GrowNet":
        split_stages, _ = train_grownet_aug(
            data["X_tr"], data["y_tr"], data["X_va"], data["y_va"], data["X_te_full"],
            n_stages=int(best_params["n_stages"]),
            hidden=int(best_params["hidden"]),
            lr=lr,
            wd=wd,
            step_size=float(best_params["step_size"]),
            bs=bs,
            patience=30,
            dropout=float(best_params.get("dropout", 0.1)),
        )
        pred_val = grownet_pred(split_stages, data["X_va"], step_size=float(best_params["step_size"]))
        pred_test_typ = grownet_pred(split_stages, data["X_te_typ"], step_size=float(best_params["step_size"]))
        pred_test_full = grownet_pred(split_stages, data["X_te_full"], step_size=float(best_params["step_size"]))

        stages_full = train_grownet_fullfit_aug(
            data["X_full"], data["y_full"],
            n_stages=int(best_params["n_stages"]),
            hidden=int(best_params["hidden"]),
            lr=lr,
            wd=wd,
            step_size=float(best_params["step_size"]),
            bs=bs,
            epochs_per_stage=int(best_params.get("epochs_per_stage", FULLFIT_GROWNET_EPOCHS_PER_STAGE)),
            dropout=float(best_params.get("dropout", 0.1)),
        )
        pred_fullfit_typ = grownet_pred(stages_full, data["X_te_typ_refit"], step_size=float(best_params["step_size"]))
        pred_fullfit_full = grownet_pred(stages_full, data["X_te_full_refit"], step_size=float(best_params["step_size"]))

        split_epochs_used = np.nan
        fullfit_epochs_used = int(best_params.get("epochs_per_stage", FULLFIT_GROWNET_EPOCHS_PER_STAGE))
    else:
        aux_w = float(DL_AUX_W.get(arch, 0.0))

        split_model = build_dl_cluster_model(arch, data["d_in"], best_params)
        split_model, _, split_ep = train_model_aug(
            split_model,
            data["X_tr"], data["y_tr"], data["X_va"], data["y_va"],
            epochs=300,
            lr=lr,
            wd=wd,
            bs=bs,
            patience=30,
            aux_w=aux_w,
        )
        if arch == "TabR":
            split_model.build_store(data["X_tr"].to(device), data["y_tr"].to(device))

        pred_val = torch_pred(split_model, data["X_va"])
        pred_test_typ = torch_pred(split_model, data["X_te_typ"])
        pred_test_full = torch_pred(split_model, data["X_te_full"])

        fullfit_epochs = int(best_params.get(
            "fullfit_epochs",
            max(FULLFIT_EPOCHS_MIN, min(FULLFIT_EPOCHS_CAP, int(split_ep) if split_ep is not None else FULLFIT_EPOCHS_DEFAULT))
        ))

        full_model = build_dl_cluster_model(arch, data["d_in"], best_params)
        full_model = train_model_fullfit_aug(
            full_model,
            data["X_full"], data["y_full"],
            epochs=fullfit_epochs,
            lr=lr,
            wd=wd,
            bs=bs,
            aux_w=aux_w,
        )
        if arch == "TabR":
            full_model.build_store(data["X_full"].to(device), data["y_full"].to(device))

        pred_fullfit_typ = torch_pred(full_model, data["X_te_typ_refit"])
        pred_fullfit_full = torch_pred(full_model, data["X_te_full_refit"])

        split_epochs_used = int(split_ep) if split_ep is not None else np.nan
        fullfit_epochs_used = int(fullfit_epochs)

    elapsed = time.time() - t0
    print(f"{arch}: расчёт выполнен за {elapsed:.1f} с")

    return {
        "split_val": pred_val,
        "split_test_typical": pred_test_typ,
        "split_test_full": pred_test_full,
        "fullfit_test_typical": pred_fullfit_typ,
        "fullfit_test_full": pred_fullfit_full,
        "split_epochs_used": split_epochs_used,
        "fullfit_epochs_used": fullfit_epochs_used,
        "time_s": elapsed,
    }

print("Вспомогательные функции cluster DL готовы.")

In [ ]:

# =========================
# Ensemble utilities
# =========================
def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, None)

def mae(y, p):
    return float(mean_absolute_error(y, clip_pred(p)))

def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, clip_pred(p))))

def mdae(y, p):
    return float(median_absolute_error(y, clip_pred(p)))

def aggregate_predictions(pred_matrix: np.ndarray, method: str, trim_frac: float = 0.1):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    if pred_matrix.ndim == 1:
        return clip_pred(pred_matrix)
    if method == "mean":
        out = pred_matrix.mean(axis=1)
    elif method == "median":
        out = np.median(pred_matrix, axis=1)
    elif method == "trimmed_mean":
        m = pred_matrix.shape[1]
        k = int(np.floor(m * trim_frac))
        if k <= 0 or 2 * k >= m:
            out = pred_matrix.mean(axis=1)
        else:
            sorted_mat = np.sort(pred_matrix, axis=1)
            out = sorted_mat[:, k:m-k].mean(axis=1)
    else:
        raise KeyError(method)
    return clip_pred(out)

def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.clip(w, 0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s

def weights_from_errors(pred_fit: pd.DataFrame, y_fit, scheme: str, temperature: float = 1.0, power: float = 1.0):
    errs = np.array([mae(y_fit, pred_fit[c].values) for c in pred_fit.columns], dtype=float)
    if scheme == "equal":
        w = np.ones(len(errs), dtype=float)
    elif scheme == "inverse_mae":
        w = 1.0 / np.maximum(errs, 1e-9)
    elif scheme == "inverse_mae_sq":
        w = 1.0 / np.maximum(errs, 1e-9) ** 2
    elif scheme == "softmax_mae":
        z = -errs / max(float(temperature), 1e-6)
        z = z - z.max()
        w = np.exp(z)
    elif scheme == "rank_power":
        order = np.argsort(errs)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(errs) + 1)
        w = 1.0 / np.power(ranks, power)
    else:
        raise KeyError(scheme)
    return normalize_weights(w)

def fit_weighted_blender(pred_fit: pd.DataFrame, y_fit, scheme: str, **kwargs):
    X = pred_fit.values.astype(float)
    if scheme in {"equal", "inverse_mae", "inverse_mae_sq", "softmax_mae", "rank_power"}:
        w = weights_from_errors(pred_fit, y_fit, scheme, **kwargs)
        return {"kind": "weights", "weights": w}
    elif scheme == "nnls":
        w, _ = nnls(X, np.asarray(y_fit, dtype=float))
        w = normalize_weights(w)
        return {"kind": "weights", "weights": w}
    elif scheme == "simplex_mae":
        n = X.shape[1]
        x0 = np.ones(n, dtype=float) / n
        bounds = [(0.0, 1.0)] * n
        cons = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}
        def objective(w):
            return mae(y_fit, X @ w)
        res = minimize(objective, x0=x0, method="SLSQP", bounds=bounds, constraints=[cons], options={"maxiter": 500, "ftol": 1e-8})
        w = normalize_weights(res.x if res.success else x0)
        return {"kind": "weights", "weights": w, "optimizer_success": bool(res.success)}
    else:
        raise KeyError(scheme)

def predict_weighted_blender(fitted, pred_df: pd.DataFrame):
    w = np.asarray(fitted["weights"], dtype=float)
    return clip_pred(pred_df.values @ w)

def build_stacker(stacker_name: str):
    if stacker_name == "linear":
        return LinearRegression()
    elif stacker_name == "positive_linear":
        return LinearRegression(positive=True)
    elif stacker_name == "ridge":
        return RidgeCV(alphas=np.logspace(-4, 4, 25))
    elif stacker_name == "lasso":
        return LassoCV(alphas=np.logspace(-4, 1, 20), cv=5, random_state=SEED, max_iter=20000)
    elif stacker_name == "elastic":
        return ElasticNetCV(alphas=np.logspace(-4, 1, 20), l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.98], cv=5, random_state=SEED, max_iter=20000)
    elif stacker_name == "bayes":
        return BayesianRidge()
    elif stacker_name == "huber":
        return HuberRegressor(max_iter=1000, epsilon=1.35)
    elif stacker_name == "quantile":
        return QuantileRegressor(quantile=0.5, alpha=1e-4, solver="highs")
    elif stacker_name == "gbr":
        return GradientBoostingRegressor(loss="absolute_error", random_state=SEED, n_estimators=300, learning_rate=0.05, max_depth=2, min_samples_leaf=5, min_samples_split=4, subsample=0.9)
    elif stacker_name == "rf":
        return RandomForestRegressor(n_estimators=500, random_state=SEED, min_samples_leaf=2, n_jobs=-1)
    elif stacker_name == "etr":
        return ExtraTreesRegressor(n_estimators=500, random_state=SEED, min_samples_leaf=2, n_jobs=-1)
    elif stacker_name == "xgb":
        if not HAS_XGB_FOR_STACKING:
            raise RuntimeError("xgboost недоступен")
        return XGBRegressor(
            objective="reg:absoluteerror",
            eval_metric="mae",
            n_estimators=500,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.0,
            reg_lambda=1.0,
            random_state=SEED,
            tree_method="hist",
            verbosity=0,
        )
    else:
        raise KeyError(stacker_name)

def fit_stacker(pred_fit: pd.DataFrame, y_fit, stacker_name: str):
    model = build_stacker(stacker_name)
    model.fit(pred_fit.values.astype(float), np.asarray(y_fit, dtype=float))
    return {"kind": "stacker", "model": model, "stacker_name": stacker_name}

def predict_stacker(fitted, pred_df: pd.DataFrame):
    pred = fitted["model"].predict(pred_df.values.astype(float))
    return clip_pred(pred)

def spec_tag(spec: dict) -> str:
    fam = spec["family"]
    models = ",".join(spec["models"])
    extra = spec.get("name", fam)
    return f"{fam}::{extra}::{models}"

def evaluate_spec(spec: dict, pred_fit_df: pd.DataFrame, y_fit, pred_sel_df: pd.DataFrame, y_sel):
    models = spec["models"]
    X_fit = pred_fit_df[models]
    X_sel = pred_sel_df[models]

    fitted = None
    if spec["family"] == "single":
        pred_sel = X_sel.iloc[:, 0].values
    elif spec["family"] == "aggregate":
        pred_sel = aggregate_predictions(X_sel.values, method=spec["agg_method"], trim_frac=spec.get("trim_frac", 0.1))
    elif spec["family"] == "weighted":
        fitted = fit_weighted_blender(X_fit, y_fit, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_sel = predict_weighted_blender(fitted, X_sel)
    elif spec["family"] == "pair_grid":
        pair_cols = X_fit.columns.tolist()
        best_w, best_fit_mae = 0.5, float("inf")
        fit_a = X_fit[pair_cols[0]].values
        fit_b = X_fit[pair_cols[1]].values
        for w in PAIR_WEIGHT_GRID:
            p = w * fit_a + (1.0 - w) * fit_b
            cur = mae(y_fit, p)
            if cur < best_fit_mae:
                best_fit_mae = cur
                best_w = float(w)
        fitted = {"kind": "pair_grid", "weight_first": best_w}
        pred_sel = best_w * X_sel[pair_cols[0]].values + (1.0 - best_w) * X_sel[pair_cols[1]].values
    elif spec["family"] == "stacker":
        fitted = fit_stacker(X_fit, y_fit, spec["stacker_name"])
        pred_sel = predict_stacker(fitted, X_sel)
    else:
        raise KeyError(spec["family"])

    row = {
        "tag": spec_tag(spec),
        "family": spec["family"],
        "name": spec.get("name", spec["family"]),
        "n_models": len(models),
        "models": models,
        "selection_MAE": mae(y_sel, pred_sel),
        "selection_RMSE": rmse(y_sel, pred_sel),
        "selection_MdAE": mdae(y_sel, pred_sel),
        "fitted_obj": fitted,
        "spec": spec,
    }
    return row

def refit_and_eval_spec(spec: dict, pred_val_df: pd.DataFrame, y_val, pred_test_typ_df: pd.DataFrame, y_test_typ, pred_test_full_df: pd.DataFrame, y_test_full):
    models = spec["models"]
    X_val = pred_val_df[models]
    X_test_typ = pred_test_typ_df[models]
    X_test_full = pred_test_full_df[models]

    if spec["family"] == "single":
        pred_typ = X_test_typ.iloc[:, 0].values
        pred_full = X_test_full.iloc[:, 0].values
        fitted = None
    elif spec["family"] == "aggregate":
        pred_typ = aggregate_predictions(X_test_typ.values, method=spec["agg_method"], trim_frac=spec.get("trim_frac", 0.1))
        pred_full = aggregate_predictions(X_test_full.values, method=spec["agg_method"], trim_frac=spec.get("trim_frac", 0.1))
        fitted = None
    elif spec["family"] == "weighted":
        fitted = fit_weighted_blender(X_val, y_val, spec["weight_scheme"], **spec.get("weight_kwargs", {}))
        pred_typ = predict_weighted_blender(fitted, X_test_typ)
        pred_full = predict_weighted_blender(fitted, X_test_full)
    elif spec["family"] == "pair_grid":
        pair_cols = X_val.columns.tolist()
        best_w, best_val_mae = 0.5, float("inf")
        for w in PAIR_WEIGHT_GRID:
            p = w * X_val[pair_cols[0]].values + (1.0 - w) * X_val[pair_cols[1]].values
            cur = mae(y_val, p)
            if cur < best_val_mae:
                best_val_mae = cur
                best_w = float(w)
        fitted = {"kind": "pair_grid", "weight_first": best_w}
        pred_typ = best_w * X_test_typ[pair_cols[0]].values + (1.0 - best_w) * X_test_typ[pair_cols[1]].values
        pred_full = best_w * X_test_full[pair_cols[0]].values + (1.0 - best_w) * X_test_full[pair_cols[1]].values
    elif spec["family"] == "stacker":
        fitted = fit_stacker(X_val, y_val, spec["stacker_name"])
        pred_typ = predict_stacker(fitted, X_test_typ)
        pred_full = predict_stacker(fitted, X_test_full)
    else:
        raise KeyError(spec["family"])

    typ_metrics = calc_reg_metrics(y_test_typ, pred_typ)
    full_metrics = calc_reg_metrics(y_test_full, pred_full)
    out = {
        "tag": spec_tag(spec),
        "family": spec["family"],
        "name": spec.get("name", spec["family"]),
        "n_models": len(models),
        "models": models,
        "MAE_typical": typ_metrics["MAE"],
        "RMSE_typical": typ_metrics["RMSE"],
        "MdAE_typical": typ_metrics["MdAE"],
        "MAE_full": full_metrics["MAE"],
        "RMSE_full": full_metrics["RMSE"],
        "MdAE_full": full_metrics["MdAE"],
        "spec": spec,
        "fitted_obj": fitted,
    }
    if fitted and fitted.get("kind") == "weights":
        out["weights"] = fitted["weights"].tolist()
    if fitted and fitted.get("kind") == "pair_grid":
        out["weight_first"] = float(fitted["weight_first"])
    if fitted and fitted.get("kind") == "stacker":
        model = fitted["model"]
        if hasattr(model, "coef_"):
            out["weights_like"] = np.ravel(model.coef_).tolist()
        elif hasattr(model, "feature_importances_"):
            out["weights_like"] = np.ravel(model.feature_importances_).tolist()
    return out

def top_models_by_fit_mae(pred_fit_df: pd.DataFrame, y_fit):
    rows = []
    for c in pred_fit_df.columns:
        rows.append((c, mae(y_fit, pred_fit_df[c].values)))
    rows.sort(key=lambda x: x[1])
    return [m for m, _ in rows], pd.DataFrame(rows, columns=["model", "blend_fit_MAE"])

def greedy_forward_mean(pred_fit_df: pd.DataFrame, y_fit, ranked_models):
    selected = [ranked_models[0]]
    best_fit = mae(y_fit, pred_fit_df[selected].mean(axis=1).values)
    history = [{"step": 1, "models": selected.copy(), "fit_MAE": best_fit}]
    remaining = ranked_models[1:].copy()
    while remaining:
        best_candidate = None
        best_candidate_mae = best_fit
        for cand in remaining:
            trial_models = selected + [cand]
            trial_mae = mae(y_fit, pred_fit_df[trial_models].mean(axis=1).values)
            if trial_mae < best_candidate_mae - 1e-9:
                best_candidate = cand
                best_candidate_mae = trial_mae
        if best_candidate is None:
            break
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        best_fit = best_candidate_mae
        history.append({"step": len(selected), "models": selected.copy(), "fit_MAE": best_fit})
    return history

def greedy_forward_simplex(pred_fit_df: pd.DataFrame, y_fit, ranked_models):
    selected = [ranked_models[0]]
    fitted = fit_weighted_blender(pred_fit_df[selected], y_fit, "simplex_mae")
    best_fit = mae(y_fit, predict_weighted_blender(fitted, pred_fit_df[selected]))
    history = [{"step": 1, "models": selected.copy(), "fit_MAE": best_fit}]
    remaining = ranked_models[1:].copy()
    while remaining:
        best_candidate = None
        best_candidate_mae = best_fit
        for cand in remaining:
            trial_models = selected + [cand]
            fitted_trial = fit_weighted_blender(pred_fit_df[trial_models], y_fit, "simplex_mae")
            trial_mae = mae(y_fit, predict_weighted_blender(fitted_trial, pred_fit_df[trial_models]))
            if trial_mae < best_candidate_mae - 1e-9:
                best_candidate = cand
                best_candidate_mae = trial_mae
        if best_candidate is None:
            break
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        best_fit = best_candidate_mae
        history.append({"step": len(selected), "models": selected.copy(), "fit_MAE": best_fit})
    return history


In [ ]:
holdout_typ_mask = (holdout_raw[TARGET_COL] < DURATION_CAP).reset_index(drop=True).values

top10_aug_data = {}
prep_rows = []

for _, row in tuned_df.iterrows():
    dl_name = row["model"]
    clusterer_name = row["clusterer"]
    cluster_params = parse_maybe_dict(row["cluster_params"])

    result_full = cluster_fit_and_assign(
        clusterer_name,
        cluster_params,
        cluster_Xtr_embed,
        cluster_Xte_embed,
    )
    if result_full is None:
        print(f"Признаки не построены: {dl_name} | {clusterer_name}")
        continue

    tr_labels_full, te_labels_full, tr_proba_full, te_proba_full, fit_mode = result_full

    tr_feat_full, te_feat_full = cluster_build_meta_features(
        tr_labels_full,
        te_labels_full,
        tr_proba=tr_proba_full,
        te_proba=te_proba_full,
    )

    Xtr_aug = pd.concat(
        [cluster_Xtr0.reset_index(drop=True), tr_feat_full.reset_index(drop=True)],
        axis=1,
    )
    Xte_full_aug = pd.concat(
        [cluster_Xte0.reset_index(drop=True), te_feat_full.reset_index(drop=True)],
        axis=1,
    )
    Xte_typ_aug = Xte_full_aug.loc[holdout_typ_mask].reset_index(drop=True)

    val_cut_local = int(len(Xtr_aug) * 0.8)

    X_tr_np = Xtr_aug.iloc[:val_cut_local].values.astype(np.float32)
    y_tr_np = cluster_ytr0[:val_cut_local].astype(np.float32)
    X_va_np = Xtr_aug.iloc[val_cut_local:].values.astype(np.float32)
    y_va_np = cluster_ytr0[val_cut_local:].astype(np.float32)

    X_te_full_np = Xte_full_aug.values.astype(np.float32)
    X_te_typ_np = Xte_typ_aug.values.astype(np.float32)

    sc_aug = StandardScaler()
    X_tr_np = sc_aug.fit_transform(X_tr_np)
    X_va_np = sc_aug.transform(X_va_np)
    X_te_full_np = sc_aug.transform(X_te_full_np)
    X_te_typ_np = sc_aug.transform(X_te_typ_np)

    for arr in [X_tr_np, X_va_np, X_te_full_np, X_te_typ_np]:
        np.nan_to_num(arr, copy=False)

    sc_full = StandardScaler()
    X_full_np = sc_full.fit_transform(Xtr_aug.values.astype(np.float32))
    X_te_full_store_np = sc_full.transform(Xte_full_aug.values.astype(np.float32))
    X_te_typ_store_np = X_te_full_store_np[holdout_typ_mask]

    np.nan_to_num(X_full_np, copy=False)
    np.nan_to_num(X_te_full_store_np, copy=False)
    np.nan_to_num(X_te_typ_store_np, copy=False)

    top10_aug_data[dl_name] = {
        "d_in": Xtr_aug.shape[1],
        "X_tr": torch.from_numpy(X_tr_np),
        "y_tr": torch.from_numpy(y_tr_np).unsqueeze(1),
        "X_va": torch.from_numpy(X_va_np),
        "y_va": torch.from_numpy(y_va_np).unsqueeze(1),
        "X_te_full": torch.from_numpy(X_te_full_np),
        "X_te_typ": torch.from_numpy(X_te_typ_np),
        "X_full": torch.from_numpy(X_full_np),
        "y_full": torch.from_numpy(cluster_ytr0.astype(np.float32)).unsqueeze(1),
        "X_te_full_refit": torch.from_numpy(X_te_full_store_np),
        "X_te_typ_refit": torch.from_numpy(X_te_typ_store_np),
        "clusterer": clusterer_name,
        "cluster_params": cluster_params,
        "fit_mode": fit_mode,
    }

    prep_rows.append({
        "model": dl_name,
        "clusterer": clusterer_name,
        "fit_mode": fit_mode,
        "d_in_aug": int(Xtr_aug.shape[1]),
        "n_train": int(X_tr_np.shape[0]),
        "n_val": int(X_va_np.shape[0]),
        "n_holdout_typical": int(X_te_typ_np.shape[0]),
        "n_holdout_full": int(X_te_full_np.shape[0]),
    })

prep_df = pd.DataFrame(prep_rows).sort_values("model").reset_index(drop=True)
display(prep_df)
prep_df.to_csv(RUN_DIR / "cluster_top10_augmented_data_manifest.csv", index=False)
prep_df.to_csv(ART_DIR / "cluster_top10_augmented_data_manifest_latest.csv", index=False)

print(f"Подготовлено моделей: {len(top10_aug_data)}")
if len(top10_aug_data) < 2:
    raise RuntimeError("Подготовлено меньше 2 моделей — ensemble stage невозможен.")

In [ ]:
model_cache_log = []
updated_rows = []

for _, row in tuned_df.iterrows():
    model_name = row["model"]
    if model_name not in top10_aug_data:
        model_cache_log.append({"model": model_name, "status": "missing_aug_data"})
        updated_rows.append(row.to_dict())
        continue

    use_cache = (
        RESUME_IF_PREDICTIONS_EXIST
        and (not FORCE_REFRESH_BASE_PREDICTIONS)
        and _have_all_cached(model_name)
    )

    if use_cache:
        print(f"Используются сохранённые предсказания: {model_name}")
        model_cache_log.append({"model": model_name, "status": "cache"})
        row_out = row.to_dict()
        updated_rows.append(row_out)
        continue

    try:
        preds = fit_and_predict_tuned_cluster_model(row, top10_aug_data[model_name])
        for kind, arr in preds.items():
            if kind in {"time_s", "split_epochs_used", "fullfit_epochs_used"}:
                continue
            _save_pred(kind, model_name, arr)

        model_cache_log.append({
            "model": model_name,
            "status": "computed",
            "time_s": preds["time_s"],
            "split_epochs_used": preds["split_epochs_used"],
            "fullfit_epochs_used": preds["fullfit_epochs_used"],
        })

        row_out = row.to_dict()
        row_out["split_epochs_used"] = preds["split_epochs_used"]
        row_out["fullfit_epochs_used"] = preds["fullfit_epochs_used"]
        updated_rows.append(row_out)

    except Exception as e:
        model_cache_log.append({
            "model": model_name,
            "status": f"error:{type(e).__name__}",
            "error": str(e),
        })
        print(f"Ошибка для {model_name}: {type(e).__name__}: {e}")
        updated_rows.append(row.to_dict())
    finally:
        cleanup_memory()

model_cache_log_df = pd.DataFrame(model_cache_log)
display(model_cache_log_df)
model_cache_log_df.to_csv(RUN_DIR / "model_cache_log.csv", index=False)
model_cache_log_df.to_csv(ART_DIR / "model_cache_log_latest.csv", index=False)

tuned_computed_df = pd.DataFrame(updated_rows)
tuned_computed_df.to_csv(RUN_DIR / "cluster_top10_tuned_summary_computed.csv", index=False)
tuned_computed_df.to_csv(ART_DIR / "cluster_top10_tuned_summary_computed_latest.csv", index=False)

# prediction matrices
val_pred_df = load_prediction_frame("split_val", SELECTED_MODELS)
test_typ_split_pred_df = load_prediction_frame("split_test_typical", SELECTED_MODELS)
test_full_split_pred_df = load_prediction_frame("split_test_full", SELECTED_MODELS)
test_typ_fullfit_pred_df = load_prediction_frame("fullfit_test_typical", SELECTED_MODELS)
test_full_fullfit_pred_df = load_prediction_frame("fullfit_test_full", SELECTED_MODELS)

available_models = sorted(
    set(val_pred_df.columns) &
    set(test_typ_fullfit_pred_df.columns) &
    set(test_full_fullfit_pred_df.columns),
    key=lambda m: SELECTED_MODELS.index(m),
)

val_pred_df = val_pred_df[available_models].copy()
test_typ_split_pred_df = test_typ_split_pred_df[available_models].copy()
test_full_split_pred_df = test_full_split_pred_df[available_models].copy()
test_typ_fullfit_pred_df = test_typ_fullfit_pred_df[available_models].copy()
test_full_fullfit_pred_df = test_full_fullfit_pred_df[available_models].copy()

print(f"Доступно моделей: {len(available_models)}")
print(available_models)

if len(available_models) < 2:
    raise RuntimeError("Для ансамблей нужно хотя бы 2 успешно восстановленные модели.")

val_pred_df.to_csv(RUN_DIR / "base_pred_split_val.csv", index=False)
val_pred_df.to_csv(ART_DIR / "base_pred_split_val_latest.csv", index=False)
test_typ_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_typical.csv", index=False)
test_typ_fullfit_pred_df.to_csv(ART_DIR / "base_pred_fullfit_test_typical_latest.csv", index=False)
test_full_fullfit_pred_df.to_csv(RUN_DIR / "base_pred_fullfit_test_full.csv", index=False)
test_full_fullfit_pred_df.to_csv(ART_DIR / "base_pred_fullfit_test_full_latest.csv", index=False)

y_val = np.asarray(y_dl_va, dtype=float)
y_test_typ = np.asarray(ytest_typical, dtype=float)
y_test_full = np.asarray(ytest_full, dtype=float)

base_leaderboard = get_base_leaderboard_from_predictions(val_pred_df, y_val)
display(base_leaderboard.head(15))
base_leaderboard.to_csv(RUN_DIR / "computed_base_leaderboard.csv", index=False)
base_leaderboard.to_csv(ART_DIR / "computed_base_leaderboard_latest.csv", index=False)

# blend_fit / blend_select
blend_cut = int(len(y_val) * BLEND_FIT_FRAC)
if blend_cut <= 0 or blend_cut >= len(y_val):
    raise ValueError("Некорректный BLEND_FIT_FRAC.")

pred_fit_df = val_pred_df.iloc[:blend_cut].copy()
pred_sel_df = val_pred_df.iloc[blend_cut:].copy()
y_fit = y_val[:blend_cut].copy()
y_sel = y_val[blend_cut:].copy()

ranked_models, fit_rank_df = top_models_by_fit_mae(pred_fit_df, y_fit)
print("Лучшие модели по blend-fit MAE:")
display(fit_rank_df.head(15))
fit_rank_df.to_csv(RUN_DIR / "blend_fit_model_ranking.csv", index=False)
fit_rank_df.to_csv(ART_DIR / "blend_fit_model_ranking_latest.csv", index=False)

# =========================
# 2) Generate ensemble specs
# =========================
specs = []

# singles
for m in ranked_models:
    specs.append({"family": "single", "name": "single", "models": [m]})

# global aggregations on all models
all_models = ranked_models.copy()
specs.extend([
    {"family": "aggregate", "name": "all_mean", "models": all_models, "agg_method": "mean"},
    {"family": "aggregate", "name": "all_median", "models": all_models, "agg_method": "median"},
    {"family": "aggregate", "name": "all_trim10", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.10},
    {"family": "aggregate", "name": "all_trim20", "models": all_models, "agg_method": "trimmed_mean", "trim_frac": 0.20},
])

# top-k prefix ensembles
if RUN_TOPK_PREFIX_ENSEMBLES:
    for k in range(2, len(ranked_models) + 1):
        topk = ranked_models[:k]
        specs.append({"family": "aggregate", "name": f"top{k}_mean", "models": topk, "agg_method": "mean"})
        specs.append({"family": "aggregate", "name": f"top{k}_median", "models": topk, "agg_method": "median"})
        if k >= 5:
            specs.append({"family": "aggregate", "name": f"top{k}_trim10", "models": topk, "agg_method": "trimmed_mean", "trim_frac": 0.10})
        if k >= 8:
            specs.append({"family": "aggregate", "name": f"top{k}_trim20", "models": topk, "agg_method": "trimmed_mean", "trim_frac": 0.20})

        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae", "models": topk, "weight_scheme": "inverse_mae"})
        specs.append({"family": "weighted", "name": f"top{k}_inverse_mae_sq", "models": topk, "weight_scheme": "inverse_mae_sq"})
        for temp in [0.25, 0.5, 1.0, 2.0, 4.0]:
            specs.append({"family": "weighted", "name": f"top{k}_softmax_t{temp}", "models": topk, "weight_scheme": "softmax_mae", "weight_kwargs": {"temperature": temp}})
        for power in [0.5, 1.0, 2.0]:
            specs.append({"family": "weighted", "name": f"top{k}_rankpow_{power}", "models": topk, "weight_scheme": "rank_power", "weight_kwargs": {"power": power}})
        specs.append({"family": "weighted", "name": f"top{k}_nnls", "models": topk, "weight_scheme": "nnls"})
        specs.append({"family": "weighted", "name": f"top{k}_simplex_mae", "models": topk, "weight_scheme": "simplex_mae"})

# all pairs
if RUN_ALL_PAIRS:
    for a, b in combinations(ranked_models, 2):
        specs.append({"family": "aggregate", "name": "pair_mean", "models": [a, b], "agg_method": "mean"})
        specs.append({"family": "pair_grid", "name": "pair_grid", "models": [a, b]})

# all triples
if RUN_ALL_TRIPLES:
    for trio in combinations(ranked_models, 3):
        specs.append({"family": "aggregate", "name": "triple_mean", "models": list(trio), "agg_method": "mean"})

# exhaustive subsets on top-N
if RUN_EXHAUSTIVE_TOPN_SUBSETS:
    topn_models = ranked_models[:min(EXHAUSTIVE_TOP_N, len(ranked_models))]
    for r in range(2, len(topn_models) + 1):
        for combo in combinations(topn_models, r):
            specs.append({"family": "aggregate", "name": f"topN{len(topn_models)}_subset_mean", "models": list(combo), "agg_method": "mean"})

# greedy forward
if RUN_GREEDY_SEARCH:
    mean_path = greedy_forward_mean(pred_fit_df[ranked_models], y_fit, ranked_models)
    simplex_path = greedy_forward_simplex(pred_fit_df[ranked_models], y_fit, ranked_models)
    for step in mean_path:
        specs.append({"family": "aggregate", "name": f"greedy_mean_step{step['step']}", "models": step["models"], "agg_method": "mean"})
    for step in simplex_path:
        specs.append({"family": "weighted", "name": f"greedy_simplex_step{step['step']}", "models": step["models"], "weight_scheme": "simplex_mae"})

# stackers
if RUN_STACKERS:
    pool_variants = {
        "top5": ranked_models[:min(5, len(ranked_models))],
        "top10": ranked_models[:min(10, len(ranked_models))],
        "all": ranked_models,
    }
    stacker_names = ["linear", "positive_linear", "ridge", "lasso", "elastic", "bayes", "huber", "quantile", "gbr", "rf", "etr"]
    if HAS_XGB_FOR_STACKING:
        stacker_names.append("xgb")
    for pool_name, pool_models in pool_variants.items():
        if len(pool_models) >= 2:
            for stk in stacker_names:
                specs.append({"family": "stacker", "name": f"{stk}_{pool_name}", "models": pool_models, "stacker_name": stk})

print(f"Всего конфигураций ансамблей: {len(specs)}")

# =========================
# 3) Evaluate specs on blend_select
# =========================
search_rows = []
for idx, spec in enumerate(specs, start=1):
    if idx % 250 == 0:
        print(f"Проверено конфигураций: {idx}/{len(specs)}")
    row = evaluate_spec(spec, pred_fit_df, y_fit, pred_sel_df, y_sel)
    search_rows.append(row)

search_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
    }
    for r in search_rows
]).sort_values(["selection_MAE", "selection_RMSE", "n_models"], kind="stable").reset_index(drop=True)

display(search_df.head(30))
search_df.to_csv(RUN_DIR / "ensemble_search_leaderboard.csv", index=False)
search_df.to_csv(ART_DIR / "ensemble_search_leaderboard_latest.csv", index=False)

search_rows_sorted = sorted(search_rows, key=lambda r: (r["selection_MAE"], r["selection_RMSE"], r["n_models"]))
top_refit_rows = search_rows_sorted[:min(REFIT_TOP_ENSEMBLES, len(search_rows_sorted))]
print(f"Конфигураций для holdout-оценки: {len(top_refit_rows)}")

In [ ]:
# =========================
# 4) Refit top ensembles on full inner-val and evaluate on holdout
# =========================
final_rows = []
for idx, row in enumerate(top_refit_rows, start=1):
    if idx % 25 == 0:
        print(f"Holdout refit: {idx}/{len(top_refit_rows)}")
    spec = row["spec"]
    out = refit_and_eval_spec(
        spec,
        pred_val_df=val_pred_df,
        y_val=y_val,
        pred_test_typ_df=test_typ_fullfit_pred_df,
        y_test_typ=y_test_typ,
        pred_test_full_df=test_full_fullfit_pred_df,
        y_test_full=y_test_full,
    )
    out["selection_MAE"] = row["selection_MAE"]
    out["selection_RMSE"] = row["selection_RMSE"]
    out["selection_MdAE"] = row["selection_MdAE"]
    final_rows.append(out)

final_df = pd.DataFrame([
    {
        "tag": r["tag"],
        "family": r["family"],
        "name": r["name"],
        "n_models": r["n_models"],
        "models": json.dumps(r["models"], ensure_ascii=False),
        "selection_MAE": r["selection_MAE"],
        "selection_RMSE": r["selection_RMSE"],
        "selection_MdAE": r["selection_MdAE"],
        "MAE_typical": r["MAE_typical"],
        "RMSE_typical": r["RMSE_typical"],
        "MdAE_typical": r["MdAE_typical"],
        "MAE_full": r["MAE_full"],
        "RMSE_full": r["RMSE_full"],
        "MdAE_full": r["MdAE_full"],
        "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        "weights_like": json.dumps(r.get("weights_like", []), ensure_ascii=False),
        "weight_first": r.get("weight_first", np.nan),
    }
    for r in final_rows
]).sort_values(["selection_MAE", "MAE_typical", "MAE_full"], kind="stable").reset_index(drop=True)

display(final_df.head(30))
final_df.to_csv(RUN_DIR / "ensemble_holdout_leaderboard.csv", index=False)
final_df.to_csv(ART_DIR / "ensemble_holdout_leaderboard_latest.csv", index=False)

best_ensemble_row = final_rows[0]
best_single_model = min(available_models, key=lambda m: mae(y_val, val_pred_df[m].values))

single_typ = calc_reg_metrics(y_test_typ, test_typ_fullfit_pred_df[best_single_model].values)
single_full = calc_reg_metrics(y_test_full, test_full_fullfit_pred_df[best_single_model].values)

selected_model_meta = tuned_df.set_index("model")[["clusterer", "cluster_params"]].to_dict(orient="index")

comparison_payload = {
    "selection_metric": "blend_select_MAE",
    "blend_fit_frac": BLEND_FIT_FRAC,
    "available_models": available_models,
    "selected_model_meta": selected_model_meta,
    "best_single_model_by_val": best_single_model,
    "best_single_model_metrics": {
        "MAE_typical": single_typ["MAE"],
        "RMSE_typical": single_typ["RMSE"],
        "MdAE_typical": single_typ["MdAE"],
        "MAE_full": single_full["MAE"],
        "RMSE_full": single_full["RMSE"],
        "MdAE_full": single_full["MdAE"],
    },
    "best_ensemble": {
        "tag": best_ensemble_row["tag"],
        "family": best_ensemble_row["family"],
        "name": best_ensemble_row["name"],
        "models": best_ensemble_row["models"],
        "selection_MAE": best_ensemble_row["selection_MAE"],
        "MAE_typical": best_ensemble_row["MAE_typical"],
        "RMSE_typical": best_ensemble_row["RMSE_typical"],
        "MdAE_typical": best_ensemble_row["MdAE_typical"],
        "MAE_full": best_ensemble_row["MAE_full"],
        "RMSE_full": best_ensemble_row["RMSE_full"],
        "MdAE_full": best_ensemble_row["MdAE_full"],
        "weights": best_ensemble_row.get("weights", []),
        "weights_like": best_ensemble_row.get("weights_like", []),
        "weight_first": best_ensemble_row.get("weight_first", None),
    },
}

display(pd.DataFrame([
    {"item": "best_single_model_by_val", "value": best_single_model},
    {"item": "best_ensemble_tag", "value": best_ensemble_row["tag"]},
    {"item": "best_ensemble_selection_MAE", "value": best_ensemble_row["selection_MAE"]},
    {"item": "best_ensemble_MAE_typical", "value": best_ensemble_row["MAE_typical"]},
    {"item": "best_ensemble_MAE_full", "value": best_ensemble_row["MAE_full"]},
]))

with open(RUN_DIR / "best_ensemble_summary.json", "w", encoding="utf-8") as f:
    json.dump(comparison_payload, f, ensure_ascii=False, indent=2)
with open(ART_DIR / "best_ensemble_summary_latest.json", "w", encoding="utf-8") as f:
    json.dump(comparison_payload, f, ensure_ascii=False, indent=2)

best_models = best_ensemble_row["models"]
best_weights = best_ensemble_row.get("weights") or best_ensemble_row.get("weights_like") or []
if len(best_weights) == len(best_models) and len(best_weights) > 0:
    wdf = pd.DataFrame({"model": best_models, "weight": best_weights})
    wdf.to_csv(RUN_DIR / "best_ensemble_weights.csv", index=False)
    wdf.to_csv(ART_DIR / "best_ensemble_weights_latest.csv", index=False)

run_manifest = {
    "run_name": RUN_NAME,
    "data_path": str(DATA_PATH),
    "cluster_tuned_summary_path": str(CLUSTER_TUNED_SUMMARY_PATH),
    "artifacts_root": str(ART_DIR),
    "run_dir": str(RUN_DIR),
    "duration_cap": DURATION_CAP,
    "train_raw_rows": int(len(train_raw)),
    "train_typical_rows": int(len(train_typical)),
    "holdout_typical_rows": int(len(holdout_typical)),
    "holdout_full_rows": int(len(holdout_raw)),
    "selected_models": available_models,
    "blend_fit_frac": BLEND_FIT_FRAC,
    "search_specs_total": int(len(specs)),
    "search_specs_refit": int(len(top_refit_rows)),
    "best_single_model_by_val": best_single_model,
    "best_ensemble_tag": best_ensemble_row["tag"],
}

with open(RUN_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, ensure_ascii=False, indent=2)
with open(ART_DIR / "run_manifest_latest.json", "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, ensure_ascii=False, indent=2)

print("Эксперимент завершён.")
print("ensemble_search_leaderboard.csv")
print("ensemble_holdout_leaderboard.csv")
print("best_ensemble_summary.json")

In [ ]:
top_plot = final_df.head(15).copy()
plt.figure(figsize=(12, 6))
plt.barh(top_plot["tag"][::-1], top_plot["MAE_typical"][::-1])
plt.xlabel("MAE_typical")
plt.ylabel("Ensemble")
plt.title("Top-15 cluster+DL ensembles by selection_MAE -> holdout typical")
plt.tight_layout()
plt.show()

top_corr_models = ranked_models[:min(10, len(ranked_models))]
corr = val_pred_df[top_corr_models].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation of validation predictions (top-10 clusterized base models)")
plt.tight_layout()
plt.show()

display(final_df.head(20))